# Week 8 Exercise — The Price is Right

**Author:** asket  
**Location:** `week8/community_contributions/asket/`

This notebook follows **Week 8** material from **day1.ipynb** through **day5.ipynb**. It uses the main repo's agents and deal framework.

**Important:** Run this notebook with the **current working directory set to `week8/`** (the folder that contains `agents/`, `deal_agent_framework.py`, and `pricer_service.py`). That way `from agents.xxx` and `from deal_agent_framework` resolve correctly.

---

## Week 8 order of play

| Day | Topic |
|-----|--------|
| Day 1 | Modal.com and SpecialistAgent |
| Day 2 | RAG, FrontierAgent, Ensemble Agent |
| Day 3 | ScannerAgent, MessengerAgent |
| Day 4 | AutonomousPlannerAgent and DealAgentFramework |
| Day 5 | The Price Is Right Finale (Gradio UI) |

## Setup — run from week8/

Ensure you are in the **week8** directory. If you opened the notebook from `community_contributions/asket/`, switch the kernel's cwd to `week8` or run the cell below to add `week8` to the path.

In [11]:
import os
import sys
from pathlib import Path

# Run from week8/ (where agents/ and deal_agent_framework.py live)
cwd = Path.cwd()
if (cwd / "agents").exists() and (cwd / "deal_agent_framework.py").exists():
    week8_dir = cwd
elif (cwd.parent.parent / "agents").exists():
    week8_dir = cwd.parent.parent
else:
    week8_dir = cwd
if str(week8_dir) not in sys.path:
    sys.path.insert(0, str(week8_dir))
# Prefer week8/agents over asket/agents (avoid loading community_contributions/asket/agents)
sys.path = [str(week8_dir)] + [p for p in sys.path if p != str(week8_dir) and "community_contributions/asket" not in p]
os.chdir(week8_dir)
print("Working directory:", os.getcwd())

from dotenv import load_dotenv
load_dotenv(override=True)

import logging
logging.getLogger().setLevel(logging.INFO)

Working directory: /Users/franckasket/Documents/GitHub/llm_engineering2/week8


### Install Week 8 dependencies (run once)

The main week8 agents use **litellm** (Preprocessor), **chromadb**, **gradio**, **modal**, etc. Run the cell below once if you see `ModuleNotFoundError: No module named 'litellm'`; then restart the kernel and re-run from Setup. On Homebrew Python (macOS), the cell uses `--user --break-system-packages`; prefer a venv if you can.

In [12]:
%pip install -q litellm chromadb gradio modal openai python-dotenv pydantic sentence-transformers beautifulsoup4 feedparser --user --break-system-packages

3092.75s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


**After installing:** Use **Kernel → Restart** (or Restart Kernel), then run from the **Setup** cell so the new packages are loaded.

---

## Day 1 — Modal and SpecialistAgent

From **day1.ipynb**: Modal tokens, HuggingFace secret, Preprocessor, and the **SpecialistAgent** that calls the fine-tuned pricer on Modal.

**Prerequisites:** Modal account, `modal token set` (or `MODAL_TOKEN_ID` / `MODAL_TOKEN_SECRET` in `.env`), HuggingFace secret in Modal, and `uv run modal deploy -m pricer_service` from week8/. **If you don't have Modal set up**, the SpecialistAgent cell will print instructions and you can skip to Day 2.

In [13]:
# Day 1: Preprocessor (rewrites product text for the pricer)
from agents.preprocessor import Preprocessor

preprocessor = Preprocessor()
description = "Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio"
rewritten = preprocessor.preprocess(description)
print("Rewritten:", rewritten[:200] if len(rewritten) > 200 else rewritten)

13:16:04 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama3.2; provider = ollama
INFO:LiteLLM:
LiteLLM completion() model= llama3.2; provider = ollama
13:16:09 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler


Rewritten: ### System:
### Title: HyperX Quadcast Microphone
### Category: Electronics
### Brand: HyperX
### Description: A high-quality USB condenser microphone for clear and detailed audio recording and stream


In [20]:
# Day 1: SpecialistAgent — calls the deployed Modal pricer (optional: requires Modal token + deploy)
# First call can take 10–15 min (container downloads Llama + fine-tuned model). Don't interrupt.
try:
    from agents.specialist_agent import SpecialistAgent
    agent = SpecialistAgent()
    print("Calling Modal pricer (first time can take 10–15 min — model download). Check modal.com for progress.")
    price = agent.price(rewritten)
    print(f"SpecialistAgent estimate: ${price:.2f}")
except Exception as e:
    if "Token missing" in str(e) or "AuthError" in type(e).__name__:
        print("Modal not configured. This cell is optional. To use SpecialistAgent:")
        print("  1. Sign up at modal.com and run: modal token new")
        print("  2. Add HuggingFace secret 'huggingface-secret' in Modal dashboard")
        print("  3. From asket folder run: ./deploy_pricer.sh (deploys pricer_service2)")
        print("You can continue with Day 2 (Chroma, FrontierAgent) if the DB is populated.")
    elif "Pricer" in str(e) and "not found" in str(e).lower():
        print("Deployed app has no Pricer class. Deploy pricer_service2: from asket run ./deploy_pricer.sh")
    elif "gated repo" in str(e).lower() or "403" in str(e):
        print("HuggingFace gated model (Llama) access denied. Use ungated pricer: from asket run ./deploy_pricer.sh --open")
    else:
        raise

INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model


Calling Modal pricer (first time can take 10–15 min — model download). Check modal.com for progress.


INFO:root:[Specialist Agent] Specialist Agent completed - predicting $24.00


SpecialistAgent estimate: $24.00


---

## Day 2 — RAG, FrontierAgent, Ensemble Agent

From **day2.ipynb**: Chroma vectorstore with product embeddings, **FrontierAgent** (RAG + frontier LLM), **NeuralNetworkAgent**, and **EnsembleAgent** that combines them.

**Prerequisites:** Chroma DB populated (e.g. run day2 cells that create `products_vectorstore` and add embeddings). Default path: `products_vectorstore` in the week8 directory. **EnsembleAgent** also needs `deep_neural_network.pth` in week8/ (produced by running the DNN training cells in week8/day2.ipynb).

In [21]:
# Day 2: Chroma collection (must exist from day2 setup)
import chromadb

DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)
collection = client.get_or_create_collection("products")
print("Collection count:", collection.count())

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Collection count: 0


In [22]:
# Day 2: FrontierAgent — RAG-based pricing using the collection
from agents.frontier_agent import FrontierAgent

frontier = FrontierAgent(collection)
estimate = frontier.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")
print(f"FrontierAgent estimate: ${estimate:.2f}")

/Users/franckasket/Library/Python/3.13/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6950.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
0.01s - Debugger warnin

FrontierAgent estimate: $279.00


In [29]:
# Day 2: EnsembleAgent — combines Specialist, Frontier, and Neural Network
# Requires deep_neural_network.pth from week8/day2.ipynb training (run day2 cells that train and save the DNN).
try:
    from agents.ensemble_agent import EnsembleAgent
    ensemble = EnsembleAgent(collection)
    estimate = ensemble.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")
    print(f"EnsembleAgent estimate: ${estimate:.2f}")
except FileNotFoundError as e:
    if "deep_neural_network.pth" in str(e):
        print("EnsembleAgent needs trained weights. Run week8/day2.ipynb and execute the cells that train the DNN and save to deep_neural_network.pth (in week8/).")
        print("Then re-run this cell. You can continue with Day 3–5 using FrontierAgent only if you prefer.")
    else:
        raise

INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6266.78it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is usin

EnsembleAgent needs trained weights. Run week8/day2.ipynb and execute the cells that train the DNN and save to deep_neural_network.pth (in week8/).
Then re-run this cell. You can continue with Day 3–5 using FrontierAgent only if you prefer.


---

## Day 3 — ScannerAgent and MessagingAgent

From **day3.ipynb**: **ScannerAgent** scrapes RSS feeds for deals and uses an LLM to select and summarize the best ones. **MessagingAgent** sends push notifications (e.g. via Pushover).

**Prerequisites:** For ScannerAgent: OpenAI (or configured) API. For MessagingAgent: `PUSHOVER_USER` and `PUSHOVER_TOKEN` in `.env` (optional).

In [30]:
# If you get "No module named 'feedparser'" below, run THIS cell first, then Kernel → Restart, then re-run from here.
%pip install -q feedparser --user --break-system-packages
print("Installed. Now: Kernel → Restart, then re-run from Setup or from Day 3.")

13529.77s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  DEPRECATION: Building 'sgmllib3k' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'sgmllib3k'. Discussion can be found at https://github.com/pypa/pip/issues/6334
Note: you may need to restart the kernel to use updated packages.
Installed. Now: Kernel → Restart, then re-run from Setup or from Day 3.


In [31]:
# Day 3: ScannerAgent — scan for deals (uses test_scan for deterministic demo without hitting RSS)
# Requires: feedparser. If ModuleNotFoundError, run the notebook's install cell then Kernel → Restart.
try:
    from agents.scanner_agent import ScannerAgent
except ModuleNotFoundError as e:
    if "feedparser" in str(e):
        print("Missing feedparser. Run the notebook's INSTALL cell (near the top), then Kernel → Restart, then re-run from Setup.")
        raise
    raise
scanner = ScannerAgent()
result = scanner.test_scan()  # or scanner.scan() for live RSS
print(f"Deals found: {len(result.deals) if result else 0}")
if result and result.deals:
    for d in result.deals[:2]:
        print(d.product_description[:80], "...", d.price, d.url)

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready


Deals found: 4
The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stu ... 178.0 https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142
The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifi ... 30.0 https://www.dealnews.com/products/Poly-Studio-P21-21-5-1080-p-LED-Personal-Meeting-Display/378335.html?iref=rss-c39


In [32]:
# Day 3: MessagingAgent — push notification (requires Pushover keys in .env)
from agents.messaging_agent import MessagingAgent

messenger = MessagingAgent()
messenger.push("Week 8 Exercise: Deal agent test message")  # or messenger.notify(desc, deal_price, estimate, url)

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


---

## Day 4 — DealAgentFramework and Planning Agent

From **day4.ipynb** and **deal_agent_framework.py**: The **DealAgentFramework** holds a Chroma collection and a **PlanningAgent** that coordinates the Scanner, Ensemble, and Messenger. Calling `run()` scans for deals, estimates values, and appends opportunities to memory.

**Prerequisites:** Chroma collection (Day 2), deployed Modal pricer (Day 1), and optionally Pushover for notifications.

In [34]:
# Day 4: DealAgentFramework — Scanner + Ensemble + Messenger
# Requires deep_neural_network.pth in week8/ (from day2.ipynb DNN training).
try:
    from deal_agent_framework import DealAgentFramework
    framework = DealAgentFramework()
    framework.init_agents_as_needed()
    opportunities = framework.run()
    print(f"Opportunities in memory: {len(opportunities)}")
    for opp in opportunities[-3:]:
        print(f"  Deal: {opp.deal.product_description[:50]}... | price={opp.deal.price} estimate={opp.estimate:.0f} discount={opp.discount:.0f}")
except FileNotFoundError as e:
    if "deep_neural_network.pth" in str(e):
        print("DealAgentFramework needs deep_neural_network.pth. Run week8/day2.ipynb and execute the cells that train the DNN and save to deep_neural_network.pth in week8/.")
        print("Then re-run this cell. You can still run Day 5 (Gradio UI) with placeholder data.")
    else:
        raise

INFO:root:[Agent Framework] Initializing Agent Framework


[2026-03-06 16:09:16 +0000] [Agents] [INFO] [Agent Framework] Initializing Agent Framework
[2026-03-06 16:09:16 +0000] [Agents] [INFO] [Agent Framework] Initializing Agent Framework


INFO:root:[Planning Agent] Planning Agent is initializing


[2026-03-06 16:09:16 +0000] [Agents] [INFO] [Planning Agent] Planning Agent is initializing
[2026-03-06 16:09:16 +0000] [Agents] [INFO] [Planning Agent] Planning Agent is initializing


INFO:root:[Scanner Agent] Scanner Agent is initializing


...

[2026-03-06 16:09:22 +0000] [Agents] [INFO] [Frontier Agent] Frontier Agent is ready
[2026-03-06 16:09:22 +0000] [Agents] [INFO] [Frontier Agent] Frontier Agent is ready


INFO:root:[Neural Network Agent] Neural Network Agent is initializing


[2026-03-06 16:09:22 +0000] [Agents] [INFO] [Neural Network Agent] Neural Network Agent is initializing
[2026-03-06 16:09:22 +0000] [Agents] [INFO] [Neural Network Agent] Neural Network Agent is initializing


INFO:root:Neural Network is using mps


[2026-03-06 16:09:23 +0000] [Agents] [INFO] Neural Network is using mps
[2026-03-06 16:09:23 +0000] [Agents] [INFO] Neural Network is using mps
DealAgentFramework needs deep_neural_network.pth. Run week8/day2.ipynb and execute the cells that train the DNN and save to deep_neural_network.pth in week8/.
Then re-run this cell. You can still run Day 5 (Gradio UI) with placeholder data.


---

## Day 5 — The Price is Right Finale (Gradio UI)

From **day5.ipynb**: Gradio app that shows a table of **Opportunities** (deals surfaced by the framework) and lets you select a row to send a push notification via the MessagingAgent.

**Prerequisites:** Same as Day 4. Run the framework at least once so there are opportunities to display (or use initial placeholder).

In [36]:
# Day 5: Gradio UI — deals table + select row to alert (as in day5.ipynb)
import gradio as gr
from deal_agent_framework import DealAgentFramework
from agents.deals import Opportunity, Deal

agent_framework = DealAgentFramework()
try:
    agent_framework.init_agents_as_needed()
except FileNotFoundError as e:
    if "deep_neural_network.pth" in str(e):
        print("Using placeholder data (run day2.ipynb to train DNN and get full framework).")
    else:
        raise

initial_deal = Deal(product_description="Example description", price=100.0, url="https://cnn.com")
initial_opportunity = Opportunity(deal=initial_deal, estimate=200.0, discount=100.0)
initial_opps = agent_framework.memory if agent_framework.memory else [initial_opportunity]

def get_table(opps):
    return [[o.deal.product_description, o.deal.price, o.estimate, o.discount, o.deal.url] for o in opps]

def do_select(opportunities, selected_index: gr.SelectData):
    if getattr(agent_framework, "planner", None) is None:
        print("Alert skipped (framework not fully loaded — need deep_neural_network.pth from day2).")
        return
    row = selected_index.index[0]
    if row < len(opportunities):
        agent_framework.planner.messenger.alert(opportunities[row])

with gr.Blocks(title="The Price is Right", fill_width=True) as ui:
    opportunities = gr.State(initial_opps)
    gr.Markdown('<div style="text-align: center; font-size: 24px;">The Price is Right — Deal Hunting Agentic AI</div>')
    gr.Markdown('<div style="text-align: center; font-size: 14px;">Deals surfaced so far (select a row to send push notification):</div>')
    opportunities_dataframe = gr.Dataframe(
        headers=["Description", "Price", "Estimate", "Discount", "URL"],
        wrap=True, row_count=10, col_count=5, max_height=400
    )
    ui.load(get_table, inputs=[opportunities], outputs=[opportunities_dataframe])
    opportunities_dataframe.select(do_select, inputs=[opportunities], outputs=[])

ui.launch(inbrowser=True)

INFO:root:[Agent Framework] Initializing Agent Framework


[2026-03-06 16:11:15 +0000] [Agents] [INFO] [Agent Framework] Initializing Agent Framework
[2026-03-06 16:11:15 +0000] [Agents] [INFO] [Agent Framework] Initializing Agent Framework
[2026-03-06 16:11:15 +0000] [Agents] [INFO] [Agent Framework] Initializing Agent Framework
[2026-03-06 16:11:15 +0000] [Agents] [INFO] [Agent Framework] Initializing Agent Framework


INFO:root:[Planning Agent] Planning Agent is initializing


[2026-03-06 16:11:15 +0000] [Agents] [INFO] [Planning Agent] Planning Agent is initializing
[2026-03-06 16:11:15 +0000] [Agents] [INFO] [Planning Agent] Planning Agent is initializing
[2026-03-06 16:11:15 +0000] [Agents] [INFO] [Planning Agent] Planning Agent is initializing
[2026-03-06 16:11:15 +0000] [Agents] [INFO] [Planning Agent] Planning Agent is initializing


INFO:root:[Scanner Agent] Scanner Agent is initializing


...

[2026-03-06 16:11:23 +0000] [Agents] [INFO] [Neural Network Agent] Neural Network Agent is initializing
[2026-03-06 16:11:23 +0000] [Agents] [INFO] [Neural Network Agent] Neural Network Agent is initializing
[2026-03-06 16:11:23 +0000] [Agents] [INFO] [Neural Network Agent] Neural Network Agent is initializing
[2026-03-06 16:11:23 +0000] [Agents] [INFO] [Neural Network Agent] Neural Network Agent is initializing


INFO:root:Neural Network is using mps


[2026-03-06 16:11:23 +0000] [Agents] [INFO] Neural Network is using mps
[2026-03-06 16:11:23 +0000] [Agents] [INFO] Neural Network is using mps
[2026-03-06 16:11:23 +0000] [Agents] [INFO] Neural Network is using mps
[2026-03-06 16:11:23 +0000] [Agents] [INFO] Neural Network is using mps
Using placeholder data (run day2.ipynb to train DNN and get full framework).


/Users/franckasket/Library/Python/3.13/lib/python/site-packages/gradio/components/dataframe.py:192: UserWarning: The `col_count` parameter is deprecated and will be removed. Please use `column_count` instead.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


---

## Summary

- **Day 1:** Modal deploy + Preprocessor + SpecialistAgent (fine-tuned pricer).
- **Day 2:** Chroma + FrontierAgent (RAG) + EnsembleAgent (Specialist + Frontier + Neural Network).
- **Day 3:** ScannerAgent (RSS → curated deals), MessagingAgent (push notifications).
- **Day 4:** DealAgentFramework coordinates Scanner → Ensemble → Messenger; `run()` appends opportunities to memory.
- **Day 5:** Gradio UI shows opportunities table; selecting a row triggers Messenger alert.

To run the full pipeline from the command line (as in day5): `uv run price_is_right.py` from the **week8/** directory.